# Qui uso il Firms NASA per fare la previsione di incendi sul territorio

## API per vedere la mia MAP_KEY per accedere ai dati via richieste API

In [ ]:
# # Let's set your map key that was emailed to you. It should look something like 'abcdef1234567890abcdef1234567890'
# MAP_KEY = '<replace with your map_key>'
MAP_KEY = 'd2678645bce596b35dcb6e57a24a82f4'

# now let's check how many transactions we have
import pandas as pd
import requests
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

## API per vedere i datasets

In [ ]:
# let's query data_availability to find out what date range is available for various datasets
# we will explain these datasets a bit later

# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{MAP_KEY}/all"
df_datasets = pd.read_csv(da_url)
display(df_datasets)


## Script per salvare tutti i dataset per iterare sopra

In [ ]:
datasets = []
for i in df_datasets["data_id"]:
    datasets.append(i)
print(datasets)

In [ ]:
print(df_datasets.columns.tolist())
display(df_datasets.head())

## Data Ingest 

In [ ]:
import pandas as pd
import time
from urllib.error import HTTPError

dfs = []

chunk_size = 2
total_days = 365

for dataset in datasets:
    # fine = oggi
    end = pd.Timestamp.today().normalize()
    start = end - pd.Timedelta(days=total_days - 1)

    current = start

    while current <= end:
        chunk_end = min(current + pd.Timedelta(days=chunk_size - 1), end)
        giorni = (chunk_end - current).days + 1

        area_url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{MAP_KEY}/{dataset}/world/{giorni}/{chunk_end.strftime('%Y-%m-%d')}"
        )

        print(f"{dataset}: {current.date()} → {chunk_end.date()}")

        try:
            df = pd.read_csv(area_url)

            # skip se vuoto (capita spesso)
            if not df.empty:
                df["dataset"] = dataset
                df["chunk_start"] = current.date()
                df["chunk_end"] = chunk_end.date()
                dfs.append(df)

        except HTTPError as e:
            print(f"❌ ERRORE {e.code} → skip")

        current = chunk_end + pd.Timedelta(days=1)
        time.sleep(0.2)

df_finale = pd.concat(dfs, ignore_index=True)

print("DONE:", len(df_finale))

In [ ]:
df_finale.shape

In [ ]:
df = df_finale.to_csv("dataset_incendi_FIRMS.csv", index=False)
df = df_finale